# Laboratório: ETL Avançado e SCD Tipo 2 com Python (Pandas)
Neste laboratório, vamos simular a lógica de um fluxo ELT. Primeiro, veremos como o código identifica alterações para gerar o histórico (SCD Tipo 2). Depois, simularemos um fluxo completo.

In [1]:
import pandas as pd
from datetime import datetime

# Simulando o Data Warehouse (Estado Atual da dim_cliente)
# sk_cliente é a nossa Surrogate Key
dados_dw = {
    'sk_cliente': [101, 102],
    'id_cliente': [1, 2],
    'nome': ['MARIA', 'JOAO'],
    'cidade': ['SAO PAULO', 'BELO HORIZONTE'],
    'status_ativo': [True, True],
    'data_inicio': ['2023-01-01', '2023-01-01'],
    'data_fim': ['9999-12-31', '9999-12-31']
}
df_dim_cliente = pd.DataFrame(dados_dw)
print("--- DATA WAREHOUSE ATUAL (Antes da Pipeline) ---")
display(df_dim_cliente)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Simulando os dados extraídos hoje da API (A Maria mudou de cidade!)
dados_api = {
    'id_cliente': [1, 2, 3],
    'nome': ['MARIA', 'JOAO', 'ANA'],
    'cidade': ['RIO DE JANEIRO', 'BELO HORIZONTE', 'PORTO ALEGRE'] # Maria mudou, Ana é nova
}
df_staging = pd.DataFrame(dados_api)
print("--- DADOS NA STAGING (Chegaram Hoje) ---")
display(df_staging)

In [ ]:
# 1. Identificar quem sofreu alteração (O "Check" do dbt Snapshot)
df_merge = df_staging.merge(
    df_dim_cliente[df_dim_cliente['status_ativo'] == True], 
    on='id_cliente', 
    how='left', 
    suffixes=('_novo', '_dw')
)

hoje = datetime.now().strftime('%Y-%m-%d')

# Condição 1: Clientes Existentes que mudaram de cidade
clientes_mudaram = df_merge[
    (df_merge['cidade_dw'].notna()) & 
    (df_merge['cidade_novo'] != df_merge['cidade_dw'])
].copy()

# Ação: Fechar os registros antigos no DW
for index, row in clientes_mudaram.iterrows():
    mask = (df_dim_cliente['id_cliente'] == row['id_cliente']) & (df_dim_cliente['status_ativo'] == True)
    df_dim_cliente.loc[mask, 'status_ativo'] = False
    df_dim_cliente.loc[mask, 'data_fim'] = hoje

# 2. Preparar os registros novos para Inserção (SCD 2)
# Pegamos clientes novos (ID não existia) + as novas versões dos clientes que mudaram
clientes_inserir = df_merge[
    (df_merge['cidade_dw'].isna()) | 
    (df_merge['cidade_novo'] != df_merge['cidade_dw'])
].copy()

novas_linhas = []
for index, row in clientes_inserir.iterrows():
    nova_sk = df_dim_cliente['sk_cliente'].max() + len(novas_linhas) + 1 # Simulando a geração de Surrogate Key
    novas_linhas.append({
        'sk_cliente': nova_sk,
        'id_cliente': row['id_cliente'],
        'nome': row['nome_novo'],
        'cidade': row['cidade_novo'],
        'status_ativo': True,
        'data_inicio': hoje,
        'data_fim': '9999-12-31'
    })

# 3. Anexar as novas linhas ao DW
df_novos_registros = pd.DataFrame(novas_linhas)
df_dim_cliente = pd.concat([df_dim_cliente, df_novos_registros], ignore_index=True)

print("--- DATA WAREHOUSE APÓS PIPELINE (SCD Tipo 2 Aplicado) ---")
display(df_dim_cliente.sort_values(by=['id_cliente', 'data_inicio']))